In [ ]:
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

OUT = Path("results/alignment")
OUT.mkdir(parents=True, exist_ok=True)
full_path = OUT / "colap_full_aligned.fasta"
mature_path = OUT / "colap_mature_aligned.fasta"
trimmed_path = OUT / "colap_trimmed_aligned.fasta"
if not full_path.exists():
    mafft = shutil.which("mafft")
    if mafft is None:
        raise SystemExit("MAFFT is not installed and results/alignment/colap_full_aligned.fasta is absent")
    with full_path.open("w", encoding="utf-8") as output:
        subprocess.run([mafft, "--thread", "4", "--auto", "--inputorder", "results/sequences/colap_clean.faa"], check=True, stdout=output)
records = list(SeqIO.parse(full_path, "fasta"))
reference = next(record for record in records if record.id.startswith("GCF_000012325.1|"))
reference_positions = []
position = 0
for column, residue in enumerate(str(reference.seq)):
    if residue != "-":
        position += 1
        reference_positions.append((column, position))
mature_start = next(column for column, residue_position in reference_positions if residue_position == 25)
signal_columns = list(range(mature_start))
mature_columns = list(range(mature_start, len(reference.seq)))
extracted = []
for record in records:
    sequence = "".join(str(record.seq)[column] for column in mature_columns)
    extracted.append(SeqRecord(Seq(sequence), id=record.id, description=record.description))
SeqIO.write(extracted, mature_path, "fasta")
if not trimmed_path.exists():
    subprocess.run([sys.executable, "-m", "clipkit", str(mature_path), "-m", "gappy", "-g", "0.9", "-o", str(trimmed_path), "-l"], check=True)
alignment = list(SeqIO.parse(trimmed_path, "fasta"))
columns = len(alignment[0].seq)
constant = 0
informative = 0
for column in range(columns):
    letters = [str(record.seq[column]) for record in alignment if str(record.seq[column]) != "-"]
    counts = {}
    for letter in letters:
        counts[letter] = counts.get(letter, 0) + 1
    states = [count for count in counts.values() if count >= 2]
    if len(counts) == 1 and len(letters) == len(alignment):
        constant += 1
    if len(states) >= 2:
        informative += 1
pd.DataFrame([{
    "sequences": len(alignment),
    "columns": columns,
    "constant_columns": constant,
    "parsimony_informative_columns": informative,
    "signal_columns_removed": len(signal_columns),
    "mature_columns": len(mature_columns),
}]).to_csv(OUT / "alignment_qc.csv", index=False)
